# 04 · The study assistant: a LangGraph workflow

**Step 4 of the Study Assistant project.** This puts Steps 1–3 together into the assistant from the proposal. It answers from the course materials with citations, says *"This isn't covered in the course materials."* instead of guessing, explains topics simply, and makes practice quizzes.

It is a LangGraph `StateGraph`, the same pattern as `campus_help_desk_rag.ipynb` from class (router → nodes → conditional edges). Code: `src/study_assistant/assistant.py`, prompts in `prompts.py`.

```
question → route ─► chitchat (canned reply)
             │ answer / simplify / quiz
          retrieve   top 8 chunks, grouped into ≤5 passages (one per document)
             │
           check     off-topic floor (similarity < 0.25) + LLM relevance check
             ├── nothing relevant ─► abstain
          generate   answer / simple explanation / quiz, citing passages as [n]
             │
            cite     drop citations to passages that weren't shown, renumber,
                     map [n] → "Lecture 9 · Module 9 …, slide 6";
                     "not covered" reply → marked as declined; no citations → warning
             │
           verify    optional support check (off by default; report Section 7)
             │
           reply     answer + sources, or "This isn't covered in the course materials."
```

**Why two safety layers?** Step 2 showed similarity scores can't separate covered from not-covered questions. So an LLM checks relevance, and the answering prompt *also* tells the model to reply "not covered" when the passages aren't enough. The Step 5 evaluation later showed that, in practice, the answering prompt does almost all of this work (report, Section 6).

This notebook uses the default system: hybrid retrieval, Llama 3.1 8B on Ollama, support check off.

From the terminal: `uv run python -m study_assistant.chat -v`

In [1]:
import warnings; warnings.filterwarnings("ignore")
from study_assistant.assistant import StudyAssistant
from study_assistant.llm import get_llm

bot = StudyAssistant(get_llm("ollama"))   # defaults: hybrid retrieval, support check off; Llama 3.1 8B, local
print(bot.mermaid())

def show(question):
    r = bot.ask(question)
    print("\n".join(f"  · {line}" for line in r.trace))
    print("\n" + r.format())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	chitchat(chitchat)
	retrieve(retrieve)
	check(check)
	abstain(abstain)
	generate(generate)
	cite(cite)
	verify(verify)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	check -.-> abstain;
	check -.-> generate;
	cite --> verify;
	generate --> cite;
	retrieve --> check;
	route -.-> chitchat;
	route -.-> retrieve;
	abstain --> __end__;
	chitchat --> __end__;
	verify --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 1. An in-scope question
The relevance check keeps three Lecture 9 slides and the GPT practical's positional-embedding section, and drops the outline slide. The answer cites only the passage it actually used.

In [2]:
show("Why do transformers need positional encodings?")

  · route: mode=answer, search='Why do transformers need positional encodings?'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.63 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 7
  ·   [2] 0.39 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 2
  ·   [3] 0.59 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 6
  ·   [4] 0.54 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 8
  ·   [5] 0.30 Lecture 9 · Module 8 - Practical: Building a GPT from Scratch, cells 21-22 (6. Token and positional embeddings)
  · check: grader says relevant=[1, 3, 4, 5]
  · generate (ollama/llama3.1:8b, 5.2s)
  · cite: 1 source(s) cited

Transformers need positional encodings because they are permutation-equivariant, meaning they don't inherently understand the order of the input tokens. This is a problem because the order of the tokens is crucial for tasks like language translation and text generation. Withou

## 2. Out of scope
*Mode collapse* sounds like GAN material, but the course never covers it. Watch the `check` line: Llama 3.1 8B, acting as the grader, marks two GAN passages as relevant. The **second layer** (the answering prompt) catches it, and the assistant declines.

In [3]:
show("What is mode collapse in GANs and how do I fix it?")

  · route: mode=answer, search='What is mode collapse in GANs and how do I fix it?'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.32 Lecture 6 · Module 6: Practical - GAN, cells 2-6 (Introduction)
  ·   [2] 0.28 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 4
  ·   [3] 0.48 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 14
  ·   [4] 0.10 Add-on lab · LLM / Agent Evaluations, cells 18-19 (Tier 3 - Semantic similarity (sentence embeddings) › 4c - Failure mode tagging)
  ·   [5] 0.46 Lecture 9 · Module 9 – Transformer Architecture and Advanced GAN, slide 15
  · check: grader says relevant=[1, 2]
  · generate (ollama/llama3.1:8b, 0.7s)
  · cite: model answered 'not covered' -> abstain

This isn't covered in the course materials.


A plainly off-topic question never reaches the LLM: its best similarity is below the floor (0.25).

In [4]:
show("How do I make sourdough bread?")

  · route: mode=answer, search='How do I make sourdough bread?'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.15 Lecture 11 · Module 11 – Application of Reinforcement Learning to Generative AI, slide 10
  ·   [2] 0.15 Lecture 6 · Module 6: Practical - GAN, cells 16-19 (Finally we will train a WGAN and after each epoch generate…)
  ·   [3] 0.05 Lecture 11 · Module 10 - Applying Reinforcement Learning to Language, cells 1-3 (Introduction)
  ·   [4] 0.20 Add-on lab · Campus Help Desk - Agentic RAG Demo, cells 7-8 (3. The generator (the "G" in RAG))
  ·   [5] 0.04 Lecture 10 · Module 10 – Basics of Reinforcement Learning (extended deck), slide 6
  · check: best similarity 0.20 below floor -> abstain
  · abstain

This isn't covered in the course materials.


**A known failure.** *Capital of France* should be declined, but the extended Lecture 11 deck uses this very question as an example prompt (slide 4: what a base model does with it). So it scores 0.43, above the floor and above some real questions, and the grader accepts that slide. The answer is faithful to the slide but is not what the student asked. Off-topic questions that match example prompts inside the course can slip through; this is listed in the report's limitations.

In [5]:
show("What is the capital of France?")

  · route: mode=answer, search='What is the capital of France?'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.43 Lecture 11 · Module 11 – Application of RL to Generative AI (extended deck), slide 4
  ·   [2] 0.36 Add-on lab · LLM / Agent Evaluations, cells 9-10 (Tier 3 - Semantic similarity (sentence embeddings))
  ·   [3] 0.22 Lecture 2 · Module 2 – Basics of Generative AI, slide 18
  ·   [4] 0.16 Lecture 4 · Module 3: Practical 2 - Convolution Intuition, cells 8-10 (For this activity we will deal with a grayscale version of…)
  ·   [5] 0.17 Add-on lab · RAG Primer (Fast) - Retrieval-Augmented Generation, cells 1-2 (Introduction)
  · check: grader says relevant=[1, 5]
  · generate (ollama/llama3.1:8b, 7.9s)
  · cite: 1 source(s) cited

The base model would predict the next token as "What is the capital of Germany?" because it continues the document in the most likely way, based on its training data [1]. However, this is not the correct answer to the question. To get the co

## 3. A paraphrase, without the course's term
This describes RLHF without naming it. When this demo was first written, the extended Lecture 11 deck wasn't indexed yet and dense retrieval missed the lecture, so the assistant gave a **false "not covered"**. With the full knowledge base and hybrid retrieval, it finds the Lecture 11 slides and answers from them.

In [6]:
show("How do you teach a chatbot to prefer the answers people like?")

  · route: mode=answer, search='How do you teach a chatbot to prefer the answers people like?'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.44 Lecture 11 · Module 11 – Application of RL to Generative AI (extended deck), slide 7
  ·   [2] 0.42 Lecture 1 · Module 1 – Introduction, slide 17
  ·   [3] 0.41 Lecture 11 · Module 11 – Application of RL to Generative AI (extended deck), slide 14
  ·   [4] 0.38 Lecture 11 · Module 11 – Application of RL to Generative AI (extended deck), slide 11
  ·   [5] 0.38 Lecture 11 · Module 11 – Application of RL to Generative AI (extended deck), slide 2
  · check: grader says relevant=[1, 3, 4]
  · generate (ollama/llama3.1:8b, 6.3s)
  · cite: 2 source(s) cited

According to the course materials, the key observation is that it is much easier to judge an answer than to write one [1]. This is the idea behind the next step: instead of asking people to write answers, ask them to rank them [1]. In the REAL-WORLD EXAMPLE, a labeller ranks the model

## 4. Explain simply
Mode detection is rule-based ("in simple terms", "ELI5", "simpler", …), like the class router: cheap and predictable. The topic words are stripped of the request phrasing before searching.

In [7]:
show("Can you explain the Bellman equation in simple terms?")

  · route: mode=simplify, search='Bellman equation'
  · retrieve (hybrid): 8 chunks -> 3 passages
  ·   [1] 0.61 Lecture 10 · Module 9 - Practical: Basics of Reinforcement Learning, cells 13-14 (2. Planning when you do know the map › 2.2 The Bellman equation, seen on one cell)
  ·   [2] 0.68 Lecture 10 · Module 10 – Basics of Reinforcement Learning, slide 14
  ·   [3] 0.40 Lecture 10 · Module 9 - Practical: Basics of Reinforcement Learning, cells 15-16 (2. Planning when you do know the map › 🔎 Check your understanding)
  · check: grader says relevant=[1, 3]
  · generate (ollama/llama3.1:8b, 20.6s)
  · cite: 1 source(s) cited

Let's break it down. The Bellman equation is a rule that helps us find the cheapest way to get out of a situation. It says: "To get out of here, take the best single step, and then behave optimally from wherever that lands you."

Think of it like this: Imagine you're in a maze, and you want to find the shortest path to the exit. The Bellman equation is like a reci

## 5. Practice quiz

In [8]:
show("Quiz me on GANs")

  · route: mode=quiz, search='GANs'
  · retrieve (hybrid): 8 chunks -> 5 passages
  ·   [1] 0.57 Lecture 6 · Module 6 – Modern Deep Learning Architectures Part 3, slide 4
  ·   [2] 0.47 Lecture 6 · Module 6: Practical - GAN, cells 2-6 (Introduction)
  ·   [3] 0.39 Lecture 1 · Module 1 – Introduction, slide 5
  ·   [4] 0.43 Lecture 1 · Module 1 – Introduction, slide 14
  ·   [5] 0.33 Lecture 5 · Module 5 – Modern Deep Learning Architectures Part 2, slide 5
  · check: grader says relevant=[1, 2]
  · generate (ollama/llama3.1:8b, 19.8s)
  · cite: 2 source(s) cited

Q1. What is the primary goal of the Generator (G) in a Generative Adversarial Network (GAN)?

A) To distinguish real data from fake data. B) To generate synthetic data from random noise. C) To evaluate samples drawn from a training set. D) To sample from the real data distribution.

Answer: B) To generate synthetic data from random noise. [1]

Q2. What is the problem with using the likelihood of the data distribution in the los

## 6. Small talk (no retrieval, no LLM call)

In [9]:
show("hello")

  · route: mode=chitchat, search=''
  · chitchat: canned reply

Hi! I'm the study assistant for Applied Generative AI. Ask me about anything from the lectures, practicals or activities, ask me to explain a topic more simply, or ask for a practice quiz (e.g. "quiz me on GANs").


## What we learned (anecdotal, 8 examples; Step 5 measures it on 50 questions)

| Observation | Implication |
|---|---|
| Answers cite the right lecture/slide; citations are remapped to a clean `[1]..[n]` source list | Citation accuracy can be scored automatically in Step 5 |
| The Llama-8B relevance check marks irrelevant passages as relevant (mode collapse, capital of France) | The answering prompt does the real declining; Step 5 confirmed the relevance check never declined in the default system |
| An off-topic question that matches an example prompt in the course gets answered | A limitation of retrieval-based scope checks; listed in the report |
| A paraphrase without the course's term needs the right lecture to be retrieved | It works now that the extended Lecture 11 deck is indexed; Step 5 found retrieval was not the bottleneck |
| flan-t5 (tried from the CLI) returns fragments without citations; the "uncited" warning flags it | flan uses a similarity threshold instead of the LLM check and stays as the weak baseline |